In [1]:
import os
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"
import tensorflow as tf
tf.keras.backend.clear_session()
import pandas as pd
from deepmreye import analyse, architecture, preprocess, train
from deepmreye.util import data_generator, model_opts, util
import numpy as np
gpus = tf.config.experimental.list_physical_devices('GPU')
tf.config.experimental.set_memory_growth(gpus[0], True)

2025-11-19 08:36:50.868757: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-19 08:36:50.951022: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-11-19 08:36:50.951076: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-11-19 08:36:50.956856: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-11-19 08:36:50.982332: I tensorflow/core/platform/cpu_feature_guar

In [2]:
def create_holdout_generators(datasets=None, train_split=0.6, train_list=None, test_list=None, **args):
    # If train_list and test_list are provided, use them directly
    if train_list is not None and test_list is not None:
        full_training_list = train_list
        full_testing_list = test_list
    else:
        # Otherwise, build from datasets
        full_training_list, full_testing_list = list(), list()
        for fn_data in datasets:
            this_file_list = [fn_data + p for p in os.listdir(fn_data)]
            np.random.shuffle(this_file_list)
            split_idx = int(train_split * len(this_file_list))
            this_training_list = this_file_list[0:split_idx]
            this_testing_list = this_file_list[split_idx:]
            full_training_list.extend(this_training_list)
            full_testing_list.extend(this_testing_list)

    # Call create_generators with the final lists
    (
        training_generator,
        testing_generator,
        single_testing_generators,
        single_testing_names,
        single_training_generators,
        single_training_names,
    ) = data_generator.create_generators(full_training_list, full_testing_list, **args)

    return (
        training_generator,
        testing_generator,
        single_testing_generators,
        single_testing_names,
        single_training_generators,
        single_training_names,
        full_testing_list,
        full_training_list,
    )


# This function converts paths of the train and test splits from 'both eyes' to 'right' or  'left' eyes
def get_divided_paths(file_list, eye="left"):
    """
    Convert a list of original .npz file paths to divided left or right paths.

    Parameters:
        file_list (list of str): original file paths
        eye (str): "left" or "right"

    Returns:
        list of str: paths to divided files
    """
    if eye not in ["left", "right"]:
        raise ValueError('eye must be "left" or "right"')

    divided_paths = []
    for orig_path in file_list:
        parts = orig_path.split(os.sep)
        dataset_folder = parts[2]  # e.g., dataset1_guided_fixations
        filename = os.path.basename(orig_path)
        base = filename.replace(".npz", "")

        divided_path = os.path.join(f"/mnt/compneuro/deepmreye_finetuning/processed_data_separate_eyes/{eye}", dataset_folder, f"{base}_{eye}.npz")
        divided_paths.append(divided_path)

    return divided_paths


In [3]:
opts = model_opts.get_opts()


In [4]:
opts['epochs']=1
opts['steps_per_epoch']=5000
opts['validation_steps']=100
print(opts)

{'kernel': 3, 'lr': 2e-05, 'filters': 32, 'multiplier': 2, 'depth': 4, 'dropout_rate': 0.1, 'num_dense': 2, 'num_fc': 1024, 'gaussian_noise': 0, 'activation': <function mish at 0x7fe06feb3520>, 'groups': 8, 'inner_timesteps': 10, 'loss_euclidean': 1, 'loss_confidence': 0.1, 'epochs': 1, 'steps_per_epoch': 5000, 'validation_steps': 100, 'train_test_split': 0.6, 'batch_size': 8, 'mixed_batches': True, 'mc_dropout': False, 'rotation_x': 5, 'rotation_y': 5, 'rotation_z': 5, 'shift': 4, 'zoom': 0.15}


In [5]:
# Getting the list of all the available processed files -- Dataset 6 is excluded
all_files=[]
for root, dirs, files in os.walk('./processed_data/'):
    for name in files:
        all_files.append(os.path.join(root,name))
        print(os.path.join(root,name))

./processed_data/dataset1_guided_fixations/sub-NDARDE877RFH.npz
./processed_data/dataset1_guided_fixations/sub-NDARAA948VFH.npz
./processed_data/dataset1_guided_fixations/sub-NDARDL305BT8.npz
./processed_data/dataset1_guided_fixations/sub-NDARAC349YUC.npz
./processed_data/dataset1_guided_fixations/sub-NDARDH086ZKK.npz
./processed_data/dataset1_guided_fixations/sub-NDARAC350BZ0.npz
./processed_data/dataset1_guided_fixations/sub-NDARDL511UND.npz
./processed_data/dataset1_guided_fixations/sub-NDARAC495TJ2.npz
./processed_data/dataset1_guided_fixations/sub-NDARDN393BJH.npz
./processed_data/dataset1_guided_fixations/sub-NDARAD615WLJ.npz
./processed_data/dataset1_guided_fixations/sub-NDARDN489EXJ.npz
./processed_data/dataset1_guided_fixations/sub-NDARAE012DGA.npz
./processed_data/dataset1_guided_fixations/sub-NDAREF164ZUJ.npz
./processed_data/dataset1_guided_fixations/sub-NDARAE358VBE.npz
./processed_data/dataset1_guided_fixations/sub-NDAREF848YWD.npz
./processed_data/dataset1_guided_fixatio

In [5]:
# Create the test-train split with 0.8. All datasets were split with the same ratio

# Group by dataset (immediate parent folder name)
datasets = {}
for f in all_files:
    dataset_name = os.path.basename(os.path.dirname(f))
    datasets.setdefault(dataset_name, []).append(f)

# 80/20 split within each dataset
train_list_rs, test_list_rs = [], []
for dataset_name, files in datasets.items():
    files = sorted(files)                # deterministic ordering
    np.random.shuffle(files)             # randomize before splitting
    split_idx = int(0.8 * len(files))    # 80% train
    train_list_rs.extend(files[:split_idx])
    test_list_rs.extend(files[split_idx:])

print("Total train:", len(train_list_rs))
print("Total test:", len(test_list_rs))

Total train: 233
Total test: 60


In [6]:
#np.savetxt('/mnt/compneuro/deepmreye_finetuning/sleepybrain/train_list.txt',train_list_rs,fmt="%s")
#np.savetxt('/mnt/compneuro/deepmreye_finetuning/sleepybrain/test_list.txt',test_list_rs,fmt="%s")

In [6]:
train_list_rs=np.loadtxt('/mnt/compneuro/deepmreye_finetuning/sleepybrain/train_list.txt',dtype=str)
test_list_rs=np.loadtxt('/mnt/compneuro/deepmreye_finetuning/sleepybrain/test_list.txt',dtype=str)

In [7]:
# Create the generators fro the model trained on both eyes 
generators_both = create_holdout_generators(train_list=train_list_rs, test_list=test_list_rs, batch_size=opts['batch_size'], augment_list=((opts['rotation_x'], opts['rotation_y'], opts['rotation_z']), opts['shift'], opts['zoom']), mixed_batches=True)

Training set (./processed_data/dataset1_guided_fixations) contains 233 subjects: 
['sub-NDARTB203DU7', 'sub-NDARGF367KVL', 'sub-NDARBN365EV3', 'sub-NDAREF893ZM8',
 'sub-NDARRX800KW8', 'sub-NDARCZ947WU5', 'sub-NDARTT272WT5', 'sub-NDARCA690EBC',
 'sub-NDARGK041TPB', 'sub-NDARAG340ERT', 'sub-NDARCU736GZ1', 'sub-NDARAH948UF0',
 'sub-NDARRW974PEF', 'sub-NDARBG188RA5', 'sub-NDARBV167RMU', 'sub-NDARND697FLK',
 'sub-NDARDN489EXJ', 'sub-NDARBJ482HJL', 'sub-NDARDY776AKH', 'sub-NDARFU395UBW',
 'sub-NDAREH852JE0', 'sub-NDARDD854GF8', 'sub-NDARCG808HDJ', 'sub-NDAREK255DEE',
 'sub-NDARDH086ZKK', 'sub-NDARLU939YX4', 'sub-NDARRZ927VC3', 'sub-NDARCG159AAP',
 'sub-NDARPD855ARC', 'sub-NDARCW611MK5', 'sub-NDARJF517HC8', 'sub-NDARRB901DWV',
 'sub-NDARFF061VUK', 'sub-NDARFG713PLR', 'sub-NDARGE536BGD', 'sub-NDARNU770PM5',
 'sub-NDARBH992ARB', 'sub-NDARDN393BJH', 'sub-NDARDA573XGG', 'sub-NDARPE752VYE',
 'sub-NDARKD064EVC', 'sub-NDAREK395BM3', 'sub-NDARHG152GZC', 'sub-NDARXE193CZ1',
 'sub-NDARGA048BD8', 'sub-N

In [37]:
# Train model
(model_both, model_inference_both) = train.train_model(dataset='both_eyes_RS-in-the-model_short', generators=generators_both, opts=opts, use_multiprocessing=False,
                                            return_untrained=False, verbose=1, save=True)

Input shape (8, 47, 29, 18, 1), Output shape (8, 10, 2)
Subjects in training set: 233, Subjects in test set: 60
5000/5000 [==============================] - 1023s 200ms/step - loss: 5.6469 - val_loss: 5.3106 - lr: 2.0000e-05


In [8]:
# Load the trained model 
generators_both = data_generator.create_generators(test_list_rs,
                                              test_list_rs)
generators_both = (*generators_both, test_list_rs, test_list_rs
              )  
(model_both, model_inference_both) = train.train_model(dataset="prediction_on_both_eyes",
                                             generators=generators_both,
                                             opts=opts,
                                             return_untrained=True)
model_inference_both.load_weights('/mnt/compneuro/deepmreye_finetuning/modelinference_both_eyes_RS-in-the-model.h5')

Training set (./processed_data/dataset1_guided_fixations) contains 60 subjects: 
['sub-NDARRL426AD5', 'sub-NDARZM903TNL', 'sub-NDARGN044FE4', 'sub-NDARAC349YUC',
 'sub-NDARFV780ABD', 'sub-NDARCG438NML', 'sub-NDARGC148JBJ', 'sub-NDARCR582GKJ',
 'sub-NDARGT551AFK', 'sub-NDAREM731BYM', 'sub-NDARRM073JKA', 'sub-NDARFB908HVX',
 'sub-NDAREK575JNM', 'sub-NDARDR439HY2', 'sub-NDARDA472JE3', 'sub-NDARCL016NHB',
 'sub-NDARKZ634RVX', 'sub-NDARHC462NGR', 'sub-NDARGJ395FKP', 'sub-NDARXR346UT5',
 'sub-NDARNK740ZVM', 'sub-NDARHG188YE9', 'sub-NDARXD345GBN', 'sub-NDARDL511UND',
 'sub-NDARPX838GCD', 'sub-NDARGB000CW8', 'sub-NDARAH503YG1', 'sub-NDARAE012DGA',
 'sub-NDARFM749LF3', 'sub-NDAREN667YTZ', 'sub-NDARVN363NNQ', 'sub-NDARVB902GA5',
 'sub-NDARWH779MZ2', 'sub-NDARDB146NWY', 'TK_17_12_14', 'FM_02_08_14', 'S20_ET',
 'S06_ET', 'S28_ET', 'S33_ET', 'S14_ET', '13_ZEJI', '17_IV5V', '21_STHZ',
 '40_OQ93', '30_V776', '16_6V9K', '31_T2OU', 'S59', 'S36', 'S28', 'S43', 'S21',
 'S42', '9081', '9070', '9053', '908

2025-11-19 08:37:51.964495: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-11-19 08:37:51.968652: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-11-19 08:37:51.971862: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: [Errno 28] No space left on device
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


In [9]:
#Evaluate the model 
(evaluation_both, scores_both) = train.evaluate_model(dataset='both_eyes', model=model_inference_both, generators=generators_both, save=True, model_description='', verbose=2, percentile_cut=80)

Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: [Errno 28] No space left on device
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: [Errno 28] No space left on device
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


2025-11-19 08:37:56.928303: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8907
2025-11-19 08:37:57.051961: W external/local_xla/xla/stream_executor/gpu/redzone_allocator.cc:322] RESOURCE_EXHAUSTED: /tmp/tempfile-compneuro-f769a202-1483386-643ee7f494c22; No space left on device
Relying on driver to perform ptx compilation. 
Modify $PATH to customize ptxas location.
This message will be only logged once.
2025-11-19 08:37:57.327757: W tensorflow/compiler/mlir/tools/kernel_gen/transforms/gpu_kernel_to_blob_pass.cc:191] Failed to compile generated PTX with ptxas. Falling back to compilation by driver.
2025-11-19 08:37:57.941856: W tensorflow/compiler/mlir/tools/kernel_gen/transforms/gpu_kernel_to_blob_pass.cc:191] Failed to compile generated PTX with ptxas. Falling back to compilation by driver.
2025-11-19 08:37:58.081034: W tensorflow/compiler/mlir/tools/kernel_gen/transforms/gpu_kernel_to_blob_pass.cc:191] Failed to compile generated PTX with ptxas. F

1 / 60 - Model Performance for ./processed_data/dataset1_guided_fixations/sub-NDARRL426AD5.npz
              Pearson             R^2-Score             Eucl. Error             
                    X     Y  Mean         X     Y  Mean        Mean Median   Std
Default         0.743 0.803 0.773     0.522 0.641 0.581       4.831  4.409 3.172
Default subTR   0.741 0.803 0.772     0.520 0.640 0.580       4.837  4.426 3.176
Refined         0.751 0.857 0.804     0.532 0.726 0.629       4.453  4.013 3.120
Refined subTR   0.749 0.852 0.801     0.532 0.719 0.626       4.467  3.929 3.127


2 / 60 - Model Performance for ./processed_data/dataset1_guided_fixations/sub-NDARZM903TNL.npz
              Pearson             R^2-Score             Eucl. Error             
                    X     Y  Mean         X     Y  Mean        Mean Median   Std
Default         0.850 0.827 0.839     0.689 0.682 0.685       3.679  2.820 3.234
Default subTR   0.850 0.827 0.838     0.689 0.682 0.685       3.676  2.811 3.23

In [11]:
fig = analyse.visualise_predictions_slider(evaluation_both, scores_both, color="rgb(0, 150, 175)", bg_color="rgb(255,255,255)",
                                   ylim=[-2, 2],)
fig.show()

FigureWidget({
    'data': [{'boxpoints': 'all',
              'fillcolor': 'rgb(180, 180, 180)',
              'line': {'color': 'rgb(0,0,0)'},
              'marker': {'color': 'rgb(0, 150, 175)',
                         'line': {'color': 'rgb(0,0,0)', 'width': 2},
                         'opacity': 0.65,
                         'size': 12},
              'name': 'Default',
              'pointpos': 0,
              'text': [participant sub-NDARRL426AD5, participant sub-NDARZM903TNL,
                       participant sub-NDARGN044FE4, participant sub-NDARAC349YUC,
                       participant sub-NDARFV780ABD, participant sub-NDARCG438NML,
                       participant sub-NDARGC148JBJ, participant sub-NDARCR582GKJ,
                       participant sub-NDARGT551AFK, participant sub-NDAREM731BYM,
                       participant sub-NDARRM073JKA, participant sub-NDARFB908HVX,
                       participant sub-NDAREK575JNM, participant sub-NDARDR439HY2,
        

In [40]:
test_list_left = get_divided_paths(test_list_rs, eye="left")
train_list_left = get_divided_paths(train_list_rs, eye="left")
test_list_right = get_divided_paths(test_list_rs, eye="right")
train_list_right = get_divided_paths(train_list_rs, eye="right")

In [41]:
generators_left_rs = create_holdout_generators(train_list=train_list_left, test_list=test_list_left, batch_size=opts['batch_size'], augment_list=((opts['rotation_x'], opts['rotation_y'], opts['rotation_z']), opts['shift'], opts['zoom']), mixed_batches=True)

Training set (/mnt/compneuro/deepmreye_finetuning/processed_data_separate_eyes/left/dataset1_guided_fixations) contains 233 subjects: 
['sub-NDARTB203DU7_left', 'sub-NDARGF367KVL_left', 'sub-NDARBN365EV3_left',
 'sub-NDAREF893ZM8_left', 'sub-NDARRX800KW8_left', 'sub-NDARCZ947WU5_left',
 'sub-NDARTT272WT5_left', 'sub-NDARCA690EBC_left', 'sub-NDARGK041TPB_left',
 'sub-NDARAG340ERT_left', 'sub-NDARCU736GZ1_left', 'sub-NDARAH948UF0_left',
 'sub-NDARRW974PEF_left', 'sub-NDARBG188RA5_left', 'sub-NDARBV167RMU_left',
 'sub-NDARND697FLK_left', 'sub-NDARDN489EXJ_left', 'sub-NDARBJ482HJL_left',
 'sub-NDARDY776AKH_left', 'sub-NDARFU395UBW_left', 'sub-NDAREH852JE0_left',
 'sub-NDARDD854GF8_left', 'sub-NDARCG808HDJ_left', 'sub-NDAREK255DEE_left',
 'sub-NDARDH086ZKK_left', 'sub-NDARLU939YX4_left', 'sub-NDARRZ927VC3_left',
 'sub-NDARCG159AAP_left', 'sub-NDARPD855ARC_left', 'sub-NDARCW611MK5_left',
 'sub-NDARJF517HC8_left', 'sub-NDARRB901DWV_left', 'sub-NDARFF061VUK_left',
 'sub-NDARFG713PLR_left', 'su

In [42]:
# Train model
(model_left, model_inference_left) = train.train_model(dataset='left_eye_RS-in-the-model', generators=generators_left_rs, opts=opts, use_multiprocessing=False,
                                            return_untrained=False, verbose=1, save=True)

Input shape (8, 24, 29, 18, 1), Output shape (8, 10, 2)
Subjects in training set: 233, Subjects in test set: 60
5000/5000 [==============================] - 955s 187ms/step - loss: 6.2870 - val_loss: 5.9464 - lr: 2.0000e-05


In [16]:
# Load the trained model 
generators_left_rs = data_generator.create_generators(test_list_rs,
                                              test_list_rs)
generators_left_rs = (*generators_left_rs, test_list_rs, test_list_rs
              )  
(model_left, model_inference_left) = train.train_model(dataset="prediction_on_left_eye",
                                             generators=generators_left_rs,
                                             opts=opts,
                                             return_untrained=True)
model_inference_left.load_weights('/mnt/compneuro/deepmreye_finetuning/modelinference_left_RS-in-the-model.h5')

Training set (./processed_data/dataset1_guided_fixations) contains 60 subjects: 
['sub-NDARRL426AD5', 'sub-NDARZM903TNL', 'sub-NDARGN044FE4', 'sub-NDARAC349YUC',
 'sub-NDARFV780ABD', 'sub-NDARCG438NML', 'sub-NDARGC148JBJ', 'sub-NDARCR582GKJ',
 'sub-NDARGT551AFK', 'sub-NDAREM731BYM', 'sub-NDARRM073JKA', 'sub-NDARFB908HVX',
 'sub-NDAREK575JNM', 'sub-NDARDR439HY2', 'sub-NDARDA472JE3', 'sub-NDARCL016NHB',
 'sub-NDARKZ634RVX', 'sub-NDARHC462NGR', 'sub-NDARGJ395FKP', 'sub-NDARXR346UT5',
 'sub-NDARNK740ZVM', 'sub-NDARHG188YE9', 'sub-NDARXD345GBN', 'sub-NDARDL511UND',
 'sub-NDARPX838GCD', 'sub-NDARGB000CW8', 'sub-NDARAH503YG1', 'sub-NDARAE012DGA',
 'sub-NDARFM749LF3', 'sub-NDAREN667YTZ', 'sub-NDARVN363NNQ', 'sub-NDARVB902GA5',
 'sub-NDARWH779MZ2', 'sub-NDARDB146NWY', 'TK_17_12_14', 'FM_02_08_14', 'S20_ET',
 'S06_ET', 'S28_ET', 'S33_ET', 'S14_ET', '13_ZEJI', '17_IV5V', '21_STHZ',
 '40_OQ93', '30_V776', '16_6V9K', '31_T2OU', 'S59', 'S36', 'S28', 'S43', 'S21',
 'S42', '9081', '9070', '9053', '908

ValueError: Cannot assign value to variable ' dense/kernel:0': Shape mismatch.The variable shape (7680, 1024), and the assigned value shape (4608, 1024) are incompatible.

In [43]:
#Evaluate the model 
(evaluation_left, scores_left) = train.evaluate_model(dataset='prediction_on_left_eye', model=model_inference_left, generators=generators_left_rs, save=True, model_description='', verbose=2, percentile_cut=80)

1 / 60 - Model Performance for /mnt/compneuro/deepmreye_finetuning/processed_data_separate_eyes/left/dataset1_guided_fixations/sub-NDARRL426AD5_left.npz
              Pearson             R^2-Score             Eucl. Error             
                    X     Y  Mean         X     Y  Mean        Mean Median   Std
Default         0.625 0.771 0.698     0.369 0.531 0.450       5.795  5.155 3.212
Default subTR   0.624 0.769 0.696     0.369 0.529 0.449       5.797  5.196 3.221
Refined         0.615 0.776 0.696     0.357 0.550 0.453       5.729  5.097 3.288
Refined subTR   0.620 0.769 0.694     0.362 0.540 0.451       5.735  5.118 3.279


2 / 60 - Model Performance for /mnt/compneuro/deepmreye_finetuning/processed_data_separate_eyes/left/dataset1_guided_fixations/sub-NDARZM903TNL_left.npz
              Pearson             R^2-Score             Eucl. Error             
                    X     Y  Mean         X     Y  Mean        Mean Median   Std
Default         0.726 0.728 0.727     0.491 

In [44]:
fig = analyse.visualise_predictions_slider(evaluation_left, scores_left, color="rgb(0, 150, 175)", bg_color="rgb(255,255,255)",
                                   ylim=[-11, 11],)
fig.show()

FigureWidget({
    'data': [{'boxpoints': 'all',
              'fillcolor': 'rgb(180, 180, 180)',
              'line': {'color': 'rgb(0,0,0)'},
              'marker': {'color': 'rgb(0, 150, 175)',
                         'line': {'color': 'rgb(0,0,0)', 'width': 2},
                         'opacity': 0.65,
                         'size': 12},
              'name': 'Default',
              'pointpos': 0,
              'text': [participant sub-NDARRL426AD5_left, participant sub-
                       NDARZM903TNL_left, participant sub-NDARGN044FE4_left,
                       participant sub-NDARAC349YUC_left, participant sub-
                       NDARFV780ABD_left, participant sub-NDARCG438NML_left,
                       participant sub-NDARGC148JBJ_left, participant sub-
                       NDARCR582GKJ_left, participant sub-NDARGT551AFK_left,
                       participant sub-NDAREM731BYM_left, participant sub-
                       NDARRM073JKA_left, participant sub-

In [45]:
generators_right_rs = create_holdout_generators(train_list=train_list_right, test_list=test_list_right, batch_size=opts['batch_size'], augment_list=((opts['rotation_x'], opts['rotation_y'], opts['rotation_z']), opts['shift'], opts['zoom']), mixed_batches=True)

Training set (/mnt/compneuro/deepmreye_finetuning/processed_data_separate_eyes/right/dataset1_guided_fixations) contains 233 subjects: 
['sub-NDARTB203DU7_right', 'sub-NDARGF367KVL_right', 'sub-NDARBN365EV3_right',
 'sub-NDAREF893ZM8_right', 'sub-NDARRX800KW8_right', 'sub-NDARCZ947WU5_right',
 'sub-NDARTT272WT5_right', 'sub-NDARCA690EBC_right', 'sub-NDARGK041TPB_right',
 'sub-NDARAG340ERT_right', 'sub-NDARCU736GZ1_right', 'sub-NDARAH948UF0_right',
 'sub-NDARRW974PEF_right', 'sub-NDARBG188RA5_right', 'sub-NDARBV167RMU_right',
 'sub-NDARND697FLK_right', 'sub-NDARDN489EXJ_right', 'sub-NDARBJ482HJL_right',
 'sub-NDARDY776AKH_right', 'sub-NDARFU395UBW_right', 'sub-NDAREH852JE0_right',
 'sub-NDARDD854GF8_right', 'sub-NDARCG808HDJ_right', 'sub-NDAREK255DEE_right',
 'sub-NDARDH086ZKK_right', 'sub-NDARLU939YX4_right', 'sub-NDARRZ927VC3_right',
 'sub-NDARCG159AAP_right', 'sub-NDARPD855ARC_right', 'sub-NDARCW611MK5_right',
 'sub-NDARJF517HC8_right', 'sub-NDARRB901DWV_right', 'sub-NDARFF061VUK_rig

In [46]:
# Train model
(model_right, model_inference_right) = train.train_model(dataset='right_eye_RS-in-the-model', generators=generators_right_rs, opts=opts, use_multiprocessing=False,
                                            return_untrained=False, verbose=1, save=True)

Input shape (8, 23, 29, 18, 1), Output shape (8, 10, 2)
Subjects in training set: 233, Subjects in test set: 60
5000/5000 [==============================] - 930s 181ms/step - loss: 6.0685 - val_loss: 5.8135 - lr: 2.0000e-05


In [48]:
#Evaluate the model 
(evaluation_right, scores_right) = train.evaluate_model(dataset='prediction_on_right_eye', model=model_inference_right, generators=generators_right_rs, save=True, model_description='', verbose=2, percentile_cut=80)

1 / 60 - Model Performance for /mnt/compneuro/deepmreye_finetuning/processed_data_separate_eyes/right/dataset1_guided_fixations/sub-NDARRL426AD5_right.npz
              Pearson             R^2-Score             Eucl. Error             
                    X     Y  Mean         X     Y  Mean        Mean Median   Std
Default         0.660 0.704 0.682     0.388 0.458 0.423       5.985  5.400 3.021
Default subTR   0.659 0.703 0.681     0.387 0.457 0.422       5.987  5.432 3.025
Refined         0.706 0.776 0.741     0.407 0.539 0.473       5.824  5.204 2.988
Refined subTR   0.708 0.772 0.740     0.411 0.532 0.471       5.795  5.138 2.987


2 / 60 - Model Performance for /mnt/compneuro/deepmreye_finetuning/processed_data_separate_eyes/right/dataset1_guided_fixations/sub-NDARZM903TNL_right.npz
              Pearson             R^2-Score             Eucl. Error             
                    X     Y  Mean         X     Y  Mean        Mean Median   Std
Default         0.792 0.761 0.777     0.

In [49]:
fig = analyse.visualise_predictions_slider(evaluation_right, scores_right, color="rgb(0, 150, 175)", bg_color="rgb(255,255,255)",
                                   ylim=[-11, 11],)
fig.show()

FigureWidget({
    'data': [{'boxpoints': 'all',
              'fillcolor': 'rgb(180, 180, 180)',
              'line': {'color': 'rgb(0,0,0)'},
              'marker': {'color': 'rgb(0, 150, 175)',
                         'line': {'color': 'rgb(0,0,0)', 'width': 2},
                         'opacity': 0.65,
                         'size': 12},
              'name': 'Default',
              'pointpos': 0,
              'text': [participant sub-NDARRL426AD5_right, participant sub-
                       NDARZM903TNL_right, participant sub-NDARGN044FE4_right,
                       participant sub-NDARAC349YUC_right, participant sub-
                       NDARFV780ABD_right, participant sub-NDARCG438NML_right,
                       participant sub-NDARGC148JBJ_right, participant sub-
                       NDARCR582GKJ_right, participant sub-NDARGT551AFK_right,
                       participant sub-NDAREM731BYM_right, participant sub-
                       NDARRM073JKA_right, parti